In [1]:
import sympy as sp
import copy
import itertools
import random
import os
import datetime
import dill
from sympy import poly
from sympy import Symbol
from sympy import Eq, solve
from sympy import groebner
from sympy.polys.orderings import ReversedGradedLexOrder

In [2]:
a, b, c, d = Symbol("a"), Symbol("b"), Symbol("c"), Symbol("d")

In [3]:
def check_number(n):
    """
    Check whether n is of a numeric type.

    Returns True if the type of n is one of these:
      - sp.core.numbers.Integer (sympy integer)
      - sp.core.numbers.Rational (sympy rational)
      - sp.core.numbers.Zero (sympy representation of zero)
      - sp.core.numbers.One (sympy representation of one)
      - sp.core.numbers.Half (sympy representation of one-half)
      - int (Python's built-in integer)
    """
    return type(n) in [
        sp.core.numbers.Integer,
        sp.core.numbers.Rational,
        sp.core.numbers.Zero,
        sp.core.numbers.One,
        sp.core.numbers.Half,
        int,
    ]


def find_xall(solution, xs):
    """
    Extract all monomials (or variables) from the expressions in the solution dictionary.

    For each entry in the solution, the function repeatedly extracts the leading monomial
    (using sp.LM with the variable list xs) until the expression reduces to a number.

    Args:
      solution: A dictionary mapping variables to symbolic expressions.
      xs: A list of sympy symbols used by sp.LM to determine the 'largest' monomial.

    Returns:
      A set of unique monomials extracted from the solution expressions.
    """
    xall = set()
    for v, sol in solution.items():
        tsol = sol  # Make a temporary copy of the expression
        # Continue extracting monomials until tsol becomes a pure number.
        while not check_number(tsol):
            # Extract the largest (leading) monomial according to xs ordering.
            xn = sp.LM(tsol, xs)
            # Add the found monomial to the set.
            xall.add(xn)
            # Remove the extracted monomial from the expression by replacing it with 0.
            tsol = tsol.subs([(xn, 0)])
    return xall


def make_one_solution(solution, xs, val="random"):
    """
    Create one concrete solution from a dictionary of symbolic expressions.

    This function first extracts all the monomials (unknowns) present in the solution using
    find_xall. If 'val' is set to 'random', it assigns each unknown a random integer between 1 and 100.
    Then, it substitutes these values into each symbolic expression to form a numeric solution.

    Args:
      solution: Dictionary mapping variables to symbolic expressions.
      xs: Variables list used for determining monomials with sp.LM.
      val: Either 'random' for random values or a list of specific values.

    Returns:
      A dictionary mapping the original keys and the encountered monomials to their evaluated numeric values.
    """
    # Extract all monomials that appear in the solution expressions.
    xall = find_xall(solution, xs)
    if val == "random":
        # Generate a random integer (from 1 to 100) for each extracted monomial.
        val = [random.randint(1, 100) for i, xn in enumerate(xall)]
    # Create the substitution configuration as a list of (monomial, value) pairs.
    subsconf = [(xn, val[i]) for i, xn in enumerate(xall)]
    # make one solution
    one_solution = {}
    # Substitute the generated values into each symbolic expression.
    for xn, sol in solution.items():
        solsubs = sol.subs(subsconf)
        one_solution[xn] = solsubs
    # Also assign the computed value to each monomial explicitly.
    for i, xn in enumerate(xall):
        one_solution[xn] = val[i]
    return one_solution


def make_poly(eqall, one_solution, xs, terms):
    """
    Construct a polynomial by combining weighted terms and substitute numerical values.

    The polynomial is built as the sum of each term multiplied by the corresponding element in xs.
    Then, a substitution based on one_solution is applied to yield a numeric polynomial.

    Args:
      eqall: (Not used in this function but may be necessary in an extended context)
      one_solution: Dictionary of substitutions mapping variables to numerical values.
      xs: List of weight symbols that multiply the terms.
      terms: List of monomials to be included in the polynomial.

    Returns:
      The resulting polynomial after performing the substitution.
    """
    out = 0
    # Sum each weighted term: xs[i] * terms[i]
    for i in range(len(terms)):
        out += xs[i] * terms[i]
    # Create the substitution configuration as a list of (variable, value) pairs.
    subsconf = [(v, sol) for v, sol in one_solution.items()]
    return out.subs(subsconf)


def sum_term(terms):
    """
    Sum a list of terms.

    Args:
      terms: A list of symbolic expressions.

    Returns:
      The sum of all expressions in the list.
    """
    out = 0
    # Iterate through each term and add it to the total sum.
    for term in terms:
        out += term
    return out


def split_poly_term(poly, pct, gconf_vars, gconf_order):
    """
    Split a polynomial into two parts based on a specified percentage of terms.

    The polynomial is first decomposed into its individual terms using sp.LM (with the provided
    variable list and term order). Then, based on the given percentage (pct), the terms are divided
    into two groups: the first group constitutes f1 and the remaining terms constitute f2.

    Args:
      poly: The polynomial to split.
      pct: Fraction (between 0 and 1) representing the portion of terms for the first group.
      gconf_vars: List of variables used by sp.LM for term extraction.
      gconf_order: The ordering setting for sp.LM.

    Returns:
      A tuple (f1, f2), where f1 is the sum of the first pct portion of terms and f2 is the sum of the rest.
    """
    terms = []
    poly_tmp = poly.copy()
    # Decompose the polynomial into its constituent terms.
    while poly_tmp != 0:
        # Extract the leading monomial based on the given variables and ordering.
        mono = sp.LM(poly_tmp, gconf_vars, order=gconf_order)
        # Retrieve the coefficient of that monomial.
        coeff = poly_tmp.coeff(mono, 1)
        term = coeff * mono
        terms.append(term)
        # Remove the extracted term from the polynomial.
        poly_tmp -= term
    # Determine the number of terms to include in the first part, ensuring at least one term.
    n = int(len(terms) * pct)
    n = max(n, 1)
    # Return the sum of the selected terms for f1 and the sum of the remaining terms for f2.
    return sum_term(terms[0:n]), sum_term(terms[n:])


def make_random_monos(gconf_vars, nmono, deg_max=8):
    """
    Generate a list of random monomials with random coefficients.

    For the given variables (gconf_vars), this function creates nmono distinct monomials.
    Each monomial is constructed by randomly assigning exponents (from 1 to deg_max) to each
    variable and multiplying by a random integer coefficient (from 1 to deg_max).

    Args:
      gconf_vars: List of sympy symbols representing the variables.
      nmono: Number of unique monomials to generate.
      deg_max: Maximum degree (exponent) for each variable.

    Returns:
      A list of randomly generated monomials.
    """

    def make_mono(mono_deg):
        """
        Construct a monomial given a tuple of degrees.

        Args:
          mono_deg: A tuple indicating the exponent for each variable in gconf_vars.

        Returns:
          A monomial computed as the product of each variable raised to its designated exponent.
        """
        out = 1
        for i, deg in enumerate(mono_deg):
            out *= gconf_vars[i] ** deg
        return out

    mono_degs = set()
    # Continue generating random exponent combinations until nmono unique ones are produced.
    while len(mono_degs) < nmono:
        mono_deg = tuple([random.randint(1, deg_max) for i in range(len(gconf_vars))])
        mono_degs.add(mono_deg)

    # For each unique exponent tuple, multiply by a random coefficient to form the monomial.
    out = [random.randint(1, deg_max) * make_mono(mono_deg) for mono_deg in mono_degs]
    print(f"random_monos = {out}")
    return out


def make_monos(gconf_vars, mono_deg):
    """
    Generate all possible monomials of a given total degree.

    Using a recursive approach, this function generates every monomial in which the sum of exponents
    equals mono_deg. It divides the total degree among the variables in gconf_vars.

    Args:
      gconf_vars: List of sympy symbols representing the variables.
      mono_deg: The total degree for which to generate monomials.

    Returns:
      A list of all monomials having the specified total degree.
    """

    def make_monos_sub(sub_vars, sub_deg, cur_mono):
        # If only one variable remains, assign all the remaining degree to it.
        if len(sub_vars) == 1:
            monos.append(cur_mono * sub_vars[0] ** sub_deg)
            return
        # Otherwise, iterate through possible exponent values for the first variable.
        for deg in range(sub_deg + 1):
            make_monos_sub(sub_vars[1:], sub_deg - deg, cur_mono * sub_vars[0] ** deg)

    monos = []
    make_monos_sub(gconf_vars, mono_deg, 1)
    return monos


def make_prob(f1, f2, p1, p2, G, gconf_order):
    # Compute the remainder n of (f1 + f2) after reduction with respect to the Groebner basis G using the specified term order
    n = sp.reduced(f1 + f2, G, order=gconf_order)[1]
    # Ensure that the remainder n is a numerical constant.
    # If it is not a number, print the problematic value and raise an error.
    if not check_number(n):
        print(f"n = {n}")
        raise ValueError("bad f1 and f2.")

    # f1 + f2 = n => f1 + p2 = -f2 + p2 + n
    # Return two expanded expressions:
    #    1. (f1 + p2) multiplied by p1.
    #    2. (-f2 + p2 + n).
    return ((f1 + p2) * p1).expand(), (-f2 + p2 + n).expand()


def make_prob_info(
    terms, eqs, gconf_vars, gconf_order, target_eq=0, G=None, max_loop=1
):
    # Helper function to extract the constant (free) part of an expression by substituting zero for all variables.
    def const_term(eq):
        return eq.subs([(var0, 0) for var0 in gconf_vars])

    # Helper function that attempts to solve a (possibly reduced) system of equations.
    # It iteratively removes one equation from the end if no solution is found,
    # and adjusts the accumulated removed expression.
    def try_solve(equations, eq_terms, eqall_removed, max_loop=1):
        len_equations = len(equations)
        solution = []
        # Loop for a maximum number of iterations.
        for loop in range(max_loop):
            remove = loop
            pos = len_equations - remove
            # Only consider the first 'pos' equations.
            tmp_equations = equations[0:pos]
            # Remove the term corresponding to the last equation in the current subset.
            eqall_removed -= eq_terms[pos - 1]
            # Attempt to solve the current subset of equations for the unknown symbols xs.
            solution = solve(tmp_equations, xs)
            if len(solution) == 0:
                # If no solution is found, continue to try with fewer equations.
                continue
        # Return the solution found (if any), the set of equations used, and the adjusted removal term.
        return solution, tmp_equations, eqall_removed

    # If a Groebner basis G is not provided, compute it for the given equations eqs.
    if not G:
        G = groebner(eqs, gconf_vars, order=gconf_order)

    # Create a list of new symbols x0, x1, ... corresponding to each term in the 'terms' list.
    xs = [Symbol(f"x{i}") for i in range(len(terms))]

    # Reduce each term in the 'terms' list using the Groebner basis G.
    terms_reduced = []
    for term in terms:
        term_reduced = sp.reduced(term, G, order=gconf_order)[1]
        terms_reduced.append(term_reduced)

    # Similarly, reduce the target equation using the Groebner basis.
    target_eq_reduced = sp.reduced(target_eq, G, order=gconf_order)[1]
    # Construct the overall equation 'eqall' by taking the negative of the non-constant (variable) part
    # of the reduced target equation.
    eqall = -(target_eq_reduced - const_term(target_eq_reduced))

    # For each reduced term, add its variable part multiplied by a new symbol to eqall.
    # This effectively constructs a linear combination of the deviations from the constant parts.
    for i, eq in enumerate(terms_reduced):
        eqall += xs[i] * (terms_reduced[i] - const_term(terms_reduced[i]))
    eqall = eqall.expand()

    # Initialize lists to accumulate the left-hand side coefficients, individual equation terms,
    # and the corresponding equation objects (equalities).
    lhs_equations = []
    eq_terms = []
    equations = []
    eqall_new = 0
    eqall_removed = 0

    # Iteratively decompose eqall by extracting its leading term.
    # sp.LM returns the leading monomial with respect to the given term order.
    while eqall != 0:
        # Get the leading monomial of eqall.
        abc_coeff = sp.LM(eqall, gconf_vars, order=gconf_order)
        # Extract the coefficient corresponding to that monomial.
        xn_coeff = eqall.coeff(abc_coeff, 1)
        # Form the complete term from the leading monomial and its coefficient.
        term = abc_coeff * xn_coeff
        # Remove this term from eqall.
        eqall -= term
        # If the coefficient is numeric (i.e., does not contain any variables),
        # subtract this term from the accumulated removed part and skip further processing.
        if check_number(xn_coeff):
            eqall_removed -= term
            continue
        else:
            # Otherwise, we want to include this term in our new (variable-dependent) aggregation.
            pass
        # Accumulate the new terms.
        eqall_new += term
        eqall = eqall.expand()
        # Store the coefficient as a left-hand side element for a new equation.
        lhs_equations.append(xn_coeff)
        # Record the term for possible later adjustments.
        eq_terms.append(term)
        # Create an equation stating that this coefficient must vanish.
        equations.append(Eq(xn_coeff, 0))

    # Attempt to solve the system of equations constructed from the non-numeric coefficients.
    solution, equations, eqall_removed = try_solve(
        equations, eq_terms, eqall_removed, max_loop=max_loop
    )
    # Return a tuple containing:
    #   - eqall_new: the aggregation of kept terms,
    #   - eqall_removed: the aggregation of removed (numeric) terms,
    #   - lhs_equations: the list of non-numeric coefficients that were set to zero,
    #   - solution: the solution from the system of equations,
    #   - xs: the newly defined symbols corresponding to terms,
    #   - terms: the original terms list,
    #   - G: the Groebner basis used.
    return eqall_new, eqall_removed, lhs_equations, solution, xs, terms, G


def make_question(
    eqs, gconf_vars, nmono_p1=5, ratio=0.5, monos=None, nmono=None, mode="normal"
):
    """
    Generates a polynomial "question" (problem) by combining given equations and monomials.

    Parameters:
        eqs: The list of equations to be used.
        gconf_vars: The configuration variables (symbols) used in the problem.
        nmono_p1: Number of monomials to sum when creating p1 (default 5).
        ratio: The ratio used when splitting the polynomial into two parts.
        monos: A list of monomials given by the user (optional).
        nmono: If monos is not provided, number of random monomials to generate.
        mode: The mode of difficulty, determining how p2 and conversions are handled.

    Returns:
        A tuple containing the constructed parts and parameters (f1, f2, gconf_vars, gconf_order,
        G, eqs, f11, f12, p1, p2) after verifying the created problem.
    """

    def check_solution(solution):
        """
        Checks if the solution dictionary is non-empty and contains at least one non-zero entry.

        Parameters:
            solution: A dictionary mapping variables to their computed solution.

        Returns:
            True if at least one value in the dictionary is non-zero, False otherwise.
        """
        if len(solution) == 0:
            return False
        for v, sol in solution.items():
            if sol != 0:
                return True
        return False

    def get_new_poly(
        terms, eqs, gconf_vars, gconf_order, target_eq=0, G=None, max_loop=1
    ):
        """
        Generates a new polynomial based on provided terms and equations.

        This function sets up the problem using make_prob_info to obtain various parameters,
        then verifies that a solution exists. It randomly generates one solution and constructs
        a polynomial using that solution.

        Parameters:
            terms: The list of monomials used to form the polynomial.
            eqs: The equations to use in generating problem information.
            gconf_vars: The configuration variables.
            gconf_order: The monomial ordering, e.g., 'grevlex'.
            target_eq: Target equation index or expression (default is 0).
            G: Grӧbner basis or additional parameter (default is None).
            max_loop: Maximum iterations for problem setup (default is 1).

        Returns:
            A tuple containing:
                new_poly: The generated polynomial.
                eqall: All equations used.
                eqall_removed: Equations (or parts) removed.
                leq: A list or number of equations (context dependent).
                solution: The computed solution dictionary.
                xs: The list of symbols in use.
                terms: (Possibly updated) terms used.
                G: The updated Grӧbner basis or related parameter.
        """
        # Set up problem information using helper function make_prob_info.
        eqall, eqall_removed, leq, solution, xs, terms, G = make_prob_info(
            terms,
            eqs,
            gconf_vars,
            gconf_order,
            target_eq=target_eq,
            G=G,
            max_loop=max_loop,
        )
        # Ensure that the solution dictionary contains at least one non-zero entry.
        if not check_solution(solution):
            raise ValueError(
                "no solution found, increasing the number of monimials may solve this error."
            )

        # Generate one random solution for the given symbols.
        one_solution = make_one_solution(
            solution, xs, val="random"
        )  # Randomly select a solution.
        # Construct the new polynomial by substituting the solution into the equations.
        new_poly = make_poly(eqall, one_solution, xs, terms)
        return new_poly, eqall, eqall_removed, leq, solution, xs, terms, G

    def convert_f_poly(terms, eqs, gconf_vars, gconf_order, f, G=None, max_loop=3):
        """
        Converts or transforms a polynomial f so that it adheres to specific constraints.

        Similar to get_new_poly, it sets parameters using make_prob_info and checks for a valid solution.
        Then, it substitutes the solution into f and adjusts the polynomial to ensure equivalence,
        verified by reducing the difference via sp.reduced.

        Parameters:
            terms: The list of monomials.
            eqs: The list of equations.
            gconf_vars: The configuration variables.
            gconf_order: The ordering of symbols (e.g., 'grevlex').
            f: The target polynomial to convert.
            G: Grӧbner basis or an additional parameter (default is None).
            max_loop: Maximum iterations for problem setup (default is 3).

        Returns:
            The converted (adjusted) polynomial that meets the required constraints.
        """
        # Set up problem information with target equation f.
        eqall, eqall_removed, leq, solution, xs, terms, G = make_prob_info(
            terms, eqs, gconf_vars, gconf_order, target_eq=f, G=G, max_loop=max_loop
        )
        if not check_solution(solution):
            raise ValueError("no solution found.")

        # Get one random valid solution.
        one_solution = make_one_solution(solution, xs, val="random")  # [1 , 20])
        # Construct a new polynomial using the solution.
        new_poly = make_poly(eqall, one_solution, xs, terms)

        # Prepare substitution configuration from the solution.
        subsconf = [(v, sol) for v, sol in one_solution.items()]
        # Adjust the new polynomial by incorporating the substituted part.
        eqall_removed_subs = eqall_removed.subs(subsconf)
        new_poly = new_poly + eqall_removed_subs
        # Compute the remainder (or adjustment term) after reducing the difference.
        eq_number = sp.reduced(f - new_poly, G, order=gconf_order)[1]
        if not check_number(eq_number):
            raise ValueError("conversion fails.")

        return new_poly + eq_number

    # Ensure that at least one of monos or nmono is provided.
    if monos is None and nmono is None:
        raise ValueError("input monos or nmono.")
    # If monos list is not provided, generate random monomials using nmono.
    if monos is not None:
        pass  # Use the provided monos directly.
    elif nmono is not None:
        monos = make_random_monos(gconf_vars, nmono)

    # Set the configuration order for monomials, commonly 'grevlex' (graded reverse lexicographic order).
    gconf_order = "grevlex"

    # Generate a new polynomial and associated data using the monos.
    new_poly, eqall, eqall_removed, leq, solution, xs, terms, G = get_new_poly(
        monos, eqs, gconf_vars, gconf_order
    )

    if mode == "normal":
        # Split the generated polynomial into two parts f1 and f2 based on the specified ratio.
        f1, f2 = split_poly_term(new_poly, ratio, gconf_vars, "grevlex")
        print(f"f1 = {f1}")
        print(f"f2 = {f2}")

        # Create a polynomial p1 by generating random monomials and summing them.
        p1 = sum_term(make_random_monos(gconf_vars, nmono_p1))
        print(f"p1 = {p1}")

        # Generate p2 as a random integer between -10 and 10.
        p2 = random.randint(-10, 10)

        # Form the problem polynomials f11 and f12 based on f1, f2, p1, p2 and the Grӧbner basis G.
        f11, f12 = make_prob(f1, f2, p1, p2, G, "grevlex")
        print(f"f11 = {f11}")
        print(f"f12 = {f12}")

        # Convert or adjust f11 and f12 to satisfy constraints using convert_f_poly.
        f11 = convert_f_poly(monos, eqs, gconf_vars, gconf_order, f11, G=G)
        f12 = convert_f_poly(monos, eqs, gconf_vars, gconf_order, f12, G=G)
        print(f"f11 = {f11}")
        print(f"f12 = {f12}")
    elif mode == "easy":
        # Similar to 'normal' mode, split the polynomial into f1 and f2.
        f1, f2 = split_poly_term(new_poly, ratio, gconf_vars, "grevlex")
        print(f"f1 = {f1}")
        print(f"f2 = {f2}")

        # Generate p1 by summing a set of random monomials.
        p1 = sum_term(make_random_monos(gconf_vars, nmono_p1))
        print(f"p1 = {p1}")

        # Generate p2 also as a sum of random monomials (different from normal mode).
        p2 = sum_term(make_random_monos(gconf_vars, nmono_p1))
        print(f"p2 = {p2}")

        # Create the problem polynomials f11 and f12.
        f11, f12 = make_prob(f1, f2, p1, p2, G, "grevlex")
        print(f"f11 = {f11}")
        print(f"f12 = {f12}")
    elif mode == "easier":
        # Once again, split new_poly into two parts f1 and f2.
        f1, f2 = split_poly_term(new_poly, ratio, gconf_vars, "grevlex")
        print(f"f1 = {f1}")
        print(f"f2 = {f2}")

        # Define p1 using a sum of random monomials.
        p1 = sum_term(make_random_monos(gconf_vars, nmono_p1))
        print(f"p1 = {p1}")

        # Here, p2 is set as a random integer between -10 and 10.
        p2 = random.randint(-10, 10)

        # Construct f11 and f12 as in the normal mode, but without further conversion.
        f11, f12 = make_prob(f1, f2, p1, p2, G, "grevlex")
        print(f"f11 = {f11}")
        print(f"f12 = {f12}")
    else:
        # Raise an error if an invalid mode is specified.
        raise ValueError(f"invalid mode {mode}.")

    # Test the validity of the problem by ensuring that f11 and f12 satisfy the relation: f11 - f12*p1 reduces to 0.
    if sp.reduced(f11 - f12 * p1, G, order=gconf_order)[1] != 0:
        raise ValueError("bad problem.")

    # Return a tuple containing all the key components of the created problem.
    conf = f1, f2, gconf_vars, gconf_order, G, eqs, f11, f12, p1, p2
    return conf

In [4]:
def make_answer_text(conf, monomials):
    # Define a helper function to format monomials (which are tuples of exponents).
    def format_monomial(mon):
        # mon is assumed to be a tuple corresponding to exponents of (a, b, c)
        parts = []
        for var, exp in zip(gconf_vars, mon):
            if exp == 0:
                continue
            elif exp == 1:
                parts.append(f"{var}")
            else:
                parts.append(f"{var}^{exp}")
        return "*".join(parts) if parts else "1"

    # Initialize
    f1, f2, gconf_vars, gconf_order, G, eqs, f11, f12, p1, p2 = conf
    f11 = sp.nsimplify(f11, rational=True)
    f12 = sp.nsimplify(f12, rational=True)
    out_str = ""

    # The ideal we reduce with is generated by the constraints.
    out_str += f"G is a groebner basis for the constraints {eqs}.\n"

    # IMPORTANT: The list must be chosen so that a polynomial Q made as a linear
    # combination of these monomials can be a candidate for Q with Q * f12 ≡ f11.
    n = len(monomials)

    # Create unknown coefficients x0, x1, …, x_{n-1} for the representation Q.
    x_syms = sp.symbols("x0:" + str(n))

    # -------------------------
    # Step (1): For each monomial m_i, compute m_i * f12 and reduce it modulo G to e_i.
    # -------------------------
    e_list = []  # This will hold the reduced expressions e_i = rem(m_i * f12, G)
    out_str += "Reduced expressions of m_i * f12 modulo G:"
    for m in monomials:
        red_prod = sp.reduced(m * f12, G, order=gconf_order)[1]
        e_list.append(red_prod)
        out_str += f" m = {m}, => reduced m*f12 modulo G = {red_prod}\n"
    out_str += "\n"

    # -------------------------
    # Step (2): Form the linear combination Sum_i x_i * e_i.
    # -------------------------
    lin_comb = sp.expand(sum(x * e for x, e in zip(x_syms, e_list)))
    out_str += "Linear combination Σ x_i·e_i:"
    out_str += f"{lin_comb}\n\n"

    # -------------------------
    # Step (3): Compute the reduced form of f11
    # and set up the equation comparing coefficients.
    # -------------------------
    f11_reduced = sp.reduced(f11, G, order=gconf_order)[1]
    out_str += "Reduced form of f11 modulo G:"
    out_str += f"{f11_reduced}\n\n"

    # We impose:
    #    lin_comb ≡ f_target_reduced
    # That is, their difference is identically zero.
    eq_poly = sp.expand(lin_comb - f11_reduced)

    # Convert the difference into a polynomial in a, b, c to compare coefficients.
    poly_eq = sp.Poly(eq_poly, gconf_vars)
    monomials_eq = poly_eq.monoms()  # list of monomials appearing
    out_str += "Monomials in the equation and their coefficients:"
    equations = []
    for mon in monomials_eq:
        coeff = poly_eq.coeff_monomial(mon)
        # coeff_rational = sp.nsimplify(coeff, rational=True)
        mon_str = format_monomial(mon)
        equations.append(coeff)
        out_str += f"  For monomial {mon_str}, coefficient: {coeff}\n"
    out_str += "\n"

    # -------------------------
    # Step (4): Solve the linear system for the unknowns x_syms.
    # -------------------------
    sol = sp.solve(equations, x_syms, dict=True)
    out_str += f"Solution for x_i: {sol}\n\n"

    # -------------------------
    # Step (5): Output the solution as Q = Sum_i x_i * m_i.
    # -------------------------
    if sol:
        sol = sol[0]  # get the solution dictionary (if multiple, one can iterate)
        Q = sp.expand(sum(sol[x] * m for x, m in zip(x_syms, monomials)))
        out_str += "The resulting polynomial Q = Σ x_i * m_i is:\n"
        out_str += f"{Q}"
    else:
        out_str += "No solution found for the given representation."

    return out_str


def make_learning_data_text(conf, f11, f12, saveprobpath=None, saveanspath=None):
    # Helper function to write text to a file at the given path.
    def write_textfile(fpath, text):
        with open(fpath, mode="w") as f:
            f.write(text)

    # Create a list of equation strings from conf[5], each appended with " = 0"
    eqs_str_list = [f"{eq} = 0" for eq in conf[5]]
    # Join all equation strings with a comma separator
    eqs_str = ", ".join(eqs_str_list)
    # Construct the problem description using the provided f11 and f12 expressions and the equations
    prob_out_str = f"Simplify ({f11})/({f12}), where {eqs_str}."
    # Replace any Python exponentiation operator '**' with the caret '^' for a more conventional format
    prob_out_str = prob_out_str.replace("**", "^")
    # Append extra instruction to solve the problem and express the answer as a polynomial
    prob_out_str += " Please solve the problem and express it with a polynomial."
    # Optional additional instruction for providing Python code if the problem cannot be solved
    # prob_out_str += " If you can not solve it, you are allowed to show an Python code to solve it."

    # Unpack the configuration (conf) tuple into respective variables.
    f1, f2, gconf_vars, gconf_order, G, eqs, f11, f12, p1, p2 = conf
    # Create a sympy polynomial from the polynomial expression p1 with respect to the given variables.
    poly_eq = sp.Poly(p1, gconf_vars)
    # Get a list of monomials (as tuples of exponents) that appear in the polynomial.
    monomials_eq = poly_eq.monoms()  # list of monomials appearing
    # Convert the tuple representation of each monomial to actual monomial expressions by taking the product of each variable raised to its corresponding exponent.
    monomials = [
        sp.prod(x**k for x, k in zip(gconf_vars, mon)) for mon in poly_eq.monoms()
    ]
    # Generate the answer text using a helper function and the list of monomials.
    ans_out_str = make_answer_text(conf, monomials)

    # Generate a unique file identifier based on the current date and time.
    now = datetime.datetime.now()
    fileid = now.strftime("%Y%m%d_%H%M%S")

    # If a path for saving the problem is provided, save the problem and answer text in a file.
    if saveprobpath:
        filename = fileid + ".txt"
        # Define anchors to delineate sections of the file for user and Copilot
        dummy_anchor1 = "## Problem"
        dummy_anchor2 = "## Answer"
        # Combine the anchors, problem description, and answer text into one string.
        save_str = "\n".join([dummy_anchor1, prob_out_str, dummy_anchor2, ans_out_str])
        # Write the combined string to the specified file under the saveprobpath directory.
        write_textfile(os.path.join(saveprobpath, filename), save_str)

    # If a path for saving the correct answer is provided, save detailed configuration variables in a separate file.
    if saveanspath:
        filename = fileid + "_correct_answer.txt"
        # Build a list of strings each describing one of the key values
        save_str_list = []
        str_gconf_vars = [str(v) for v in gconf_vars]
        save_str_list.append("import sympy as sp\n" + ", ".join(str_gconf_vars) + " = sp.symbols(\"" + " ".join(str_gconf_vars) + "\")")
        save_str_list.append(f"f1 = {f1}")
        save_str_list.append(f"f2 = {f2}")
        save_str_list.append(f"p1 = {p1}")
        save_str_list.append(f"p2 = {p2}")
        save_str_list.append(f"f11 = {f11}")
        save_str_list.append(f"f12 = {f12}")
        save_str_list.append(f"eqs = {eqs}")
        save_str_list.append(f"gconf_vars = {gconf_vars}")
        # Join the list into a single string with newlines separating each configuration detail.
        save_str = "\n".join(save_str_list)
        # Write the configuration details to the correct answer file in the saveanspath directory.
        write_textfile(os.path.join(saveanspath, filename), save_str)

    # Finally, print the problem description to the console.
    print(prob_out_str)

In [5]:
monos_all = make_monos([a, b, c], 2)
eqs = [a**1 + b**1 + c**1 - 1, a**2 + b**2 + c**2 - 4]

gconf_vars = [a, b, c]
conf = make_question(
    eqs, gconf_vars, nmono_p1=2, ratio=0.4, monos=monos_all, mode="easy"
)
f1, f2, gconf_vars, gconf_order, G, eqs, f11, f12, p1, p2 = conf

f1 = 35*a**2 + 54*a*b
f2 = 54*a*c + 35*b**2 + 54*b*c + 35*c**2
random_monos = [4*a*b*c**4, 8*a**4*b**7*c**3]
p1 = 8*a**4*b**7*c**3 + 4*a*b*c**4
random_monos = [4*a**7*b**4*c**6, a**2*b**6*c**3]
p2 = 4*a**7*b**4*c**6 + a**2*b**6*c**3
f11 = 32*a**11*b**11*c**9 + 16*a**8*b**5*c**10 + 8*a**6*b**13*c**6 + 280*a**6*b**7*c**3 + 432*a**5*b**8*c**3 + 4*a**3*b**7*c**7 + 140*a**3*b*c**4 + 216*a**2*b**2*c**4
f12 = 4*a**7*b**4*c**6 + a**2*b**6*c**3 - 54*a*c - 35*b**2 - 54*b*c - 35*c**2 + 59


In [6]:
# Save a problem text file in "./polynomial" directory.
# Save an answer parameter file for verification in "./polynomial_result" directory.
make_learning_data_text(
    conf,
    f11,
    f12,
    saveprobpath="./polynomial",
    saveanspath="./polynomial_result",
)

Simplify (32*a^11*b^11*c^9 + 16*a^8*b^5*c^10 + 8*a^6*b^13*c^6 + 280*a^6*b^7*c^3 + 432*a^5*b^8*c^3 + 4*a^3*b^7*c^7 + 140*a^3*b*c^4 + 216*a^2*b^2*c^4)/(4*a^7*b^4*c^6 + a^2*b^6*c^3 - 54*a*c - 35*b^2 - 54*b*c - 35*c^2 + 59), where a + b + c - 1 = 0, a^2 + b^2 + c^2 - 4 = 0. Please solve the problem and express it with a polynomial.
